**Isolation Forest** is an unsupervised machine learning algorithm specifically designed for anomaly detection. Unlike traditional methods that profile normal data points to find deviations, this algorithm explicitly targets and isolates anomalies from the start.

The core philosophy rests on two simple assumptions: anomalies are **rare** and they have **significantly different values** compared to normal data. Because of these traits, anomalies can be isolated much faster and with fewer decisions than standard data points.

# Key Terminologies

- **Isolation Tree (iTree):** A randomized binary tree built by repeatedly selecting a random feature and a random split threshold to isolate individual data points.

- **Path Length $h(x)$:** The number of edges sample $x$ traverses from the root node down to its terminating leaf node in a single iTree. Short paths indicate anomalies; long paths indicate inliers.

- **Average Path Length $E(h(x))$:** The expected path length of sample $x$ calculated by averaging $h(x)$ across all trees in the forest ensemble.

- **Normalization Factor $c(n)$:** The average path length of an unsuccessful search in a Binary Search Tree (BST) constructed with $n$ nodes. It serves as the mathematical baseline to standardize path lengths regardless of subsample size.

- **Anomaly Score $s(x, n)$:** A standardized metric between $0$ and $1$ calculated as $s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$. Scores $\ge 0.6$ flag anomalies, while scores $< 0.5$ indicate normal inliers.

- **Subsampling Size ($n$ or max_samples):** The size of the random subset of data (typically $n = 256$) drawn without replacement to train each tree. Subsampling boosts training speed while mitigating masking and swamping.

- **Masking:** An anomaly detection pitfall where a large group of anomalies cluster close to one another, making them appear as a normal dense region.

- **Swamping:** An anomaly detection pitfall where normal data points located close to anomalies are incorrectly flagged as anomalies.

- **Tree Height Limit ($h_{\text{max}}$)**: The max depth threshold (typically $\lceil \log_2(n) \rceil$) where tree construction stops. Since anomalies isolate quickly at shallow depths, growing branches beyond this limit yields minimal extra diagnostic value.

# Isolation Tree

## Mechanism

1. Pick the Random feature and then find its max and min.
2. Select a cut-off value between min and max.
3. all normal point goes to left child while anomaly goes to right child.
4. Repeat the process.
5. For Anomaly, we look at the Path Length (the number of stsp it took to isolate each row)

## Mathematical Example

1. **Dataset**

    |Income|Age|
    |---|---|
    |50| 35|
    |55| 38|
    |48| 32|
    |52| 40|
    |58| 36|
    |200| 12|

2. **Building Isolation Tree**

    **Depth 0**

    -  The tree randomly picks the Income feature.
    - The current minimum Income is 48, and the maximum is 200.
    - The tree randomly selects a cut-off value between 48 and 200, landing on 71.4.
    - The Partition:
        - Left Child (< 71.4): Rows [0, 1, 2, 3, 4] (All normal points go here)
        - Right Child (≥ 71.4): Row [5]
    
    **Depth 1**
    - The tree randomly picks Income again.
    - The minimum Income is 48, maximum is 58.
    -  It randomly slices at 56.7.
    - The Partition:
        - Left Child (< 56.7): Rows [0, 1, 2, 3]
        - Right Child (≥ 56.7): Row [4]

    **Depth 2**

    - It picks Income a third time.
    - Minimum is 48, maximum is 55
    - It chooses 51.7
    - The Partition:
        - Left Child (< 51.7): Rows [0, 2]
        - Right Child (≥ 51.7): Rows [1, 3]

    **Depth 3**

    - Slicing [0, 2] on Income at 48.7 separates Row 2 (< 48.7) and Row 0 (≥ 48.7).
    - Slicing [1, 3] on Age at 39.8 separates Row 1 (< 39.8) and Row 3 (≥ 39.8). 

3. **Final Path Length Summary**

    - Row 5 (Anomaly): Path Length = 1
    - Row 4 (Normal): Path Length = 2
    - Row 0 (Normal): Path Length = 4
    - Row 2 (Normal): Path Length = 4
    - Row 1 (Normal): Path Length = 4
    - Row 3 (Normal): Path Length = 4

    Row 5 is **Anomaly**.


# Core Intuition

1. **Sub-Sampling**

    Instead of feeding the entire dataset into every tree, the forest takes a subset of the data for each tree.
    
    - **The Process:** If you have 1 million rows of data, the algorithm pulls a random sub-sample (usually 256 rows) to build Tree 1. It pulls a different random 256 rows for Tree 2, and so on.
    
    - **Why it matters:** This prevents swamping (too much normal data burying the anomaly) and masking (anomalies grouping together to look like a normal cluster). It also makes training incredibly fast.


2. **Parallel Training (Growing the Forest)**
    
    The forest independently grows an ensemble of Isolation Trees (typically 100 to 200 trees).

    - Each tree is built using the completely random splitting mechanism you learned earlier.
    - Because the trees don't rely on each other, they can be built at the exact same time (in parallel), maximizing computer processing power.
    - The trees are grown until every single point in their 256-row sample is isolated.

3. **Averaging Path Lengths (The Voting Phase)**
    
    Once the forest is built, it is ready to score data points. To score a specific data point, the forest passes that point down every single tree in the ensemble.

    - The algorithm records the Path Length h(x) (number of splits) for that point in Tree 1, Tree 2, ..., Tree 100.
    - It calculates the Mean Path Length E(h(x)) across the entire forest.

4. **Computing the Universal Anomaly Score**

    A raw average path length isn't very useful on its own because its meaning changes depending on the size of your data sub-sample. To fix this, the forest converts the average path length into a standardized **Anomaly Score s(x, n)** between **0 and 1** using a mathematical formula:

    $$s(x,n)=2^{-\frac{E(h(x))}{c(n)}}$$
    
    Where:
    - E(h(x)) is the point's average path length across your forest.
    - c(n) is the mathematically expected average path length for a normal point in a tree of that size (n). It acts as a baseline.

    $$c(n)=2\ln (n-1)+0.5772156649-\frac{2(n-1)}{n}$$


## Mathematical Example

1. **Establish the Baseline c(n)**
    $$c(n)=2\ln (n-1)+0.5772156649-\frac{2(n-1)}{n}$$

    Let's use the standard sub-sample size of $n = 256$ points per tree:
    $$c(n) = 9.6675$$
    This means, on average, a completely normal, average data point expects to take about 9.67 splits to be isolated in these trees.

2. Scoring Point A (The Blatant Outlier)

    **Dataset**

    |Income|Age|
    |---|---|
    |50| 35|
    |55| 38|
    |48| 32|
    |52| 40|
    |58| 36|
    |200| 12|

    Tree selected row 5:
    - In 90 trees, it gets isolated on the very first split: path length = 1
    - In 10 trees, a bad random split happens, and it takes a few more steps: path length = 3

    **Calculate Mean Path Length $E(h(x))$:**
    $E(h(x))=\frac{(90\times 1)+(10\times 3)}{100}=\frac{90+30}{100}=\mathbf{1.2}$$

    **Calculate Anomaly Score $s(x, n)$:**

    $$s(x,n)=2^{-\frac{E(h(x))}{c(n)}}=2^{-\frac{1.2}{9.6675}}=2^{-0.1241}\approx \mathbf{0.918}$$

    **Result for Point A:** A score of **0.92** is very close to **1**. The algorithm confidently flags this point as an **anomaly**.

3. Scoring Point B (The In-Cluster Normal Point)
    Tree selected row 1:
    - Across the 100 trees, its path lengths vary between $$10, 12, 14, \text{ and } 15$$.
    - Let's say its calculated average path length across the forest is 13.5.

    $$E(h(x))=\mathbf{13.5}$$

    $$s(x,n)=2^{-\frac{13.5}{9.6675}}=2^{-1.3964}\approx \mathbf{0.380}$$
    
    - **Result for Point B:** A score of **0.38** is well below **0.5**. The algorithm confidently classifies this point as **normal data**.




# Python Implementation

In [ ]:
import numpy as np


class IsolationTreeNode:

  def __init__(self, left=None, right=None, feature=None, split=None, size=0):
    self.left = left
    self.right = right
    self.feature = feature
    self.split = split
    self.size = size
    self.is_leaf = left is None and right is None


class IsolationForest:

  def __init__(self, n_estimators=100, max_samples=256):
    self.n_estimators = n_estimators
    self.max_samples = max_samples
    self.trees = []
    self.max_depth = 0
    self.subsample_size = 0
    self.c_factor = 1.0

  def _c(self, n):
    """Calculates expected BST average path length c(n)."""
    if n <= 1:
      return 0.0
    if n == 2:
      return 1.0
    return 2.0 * (np.log(n - 1) + 0.5772156649) - (2.0 * (n - 1) / n)

  def _build_tree(self, X, current_depth):
    n_samples, n_features = X.shape

    if current_depth >= self.max_depth or n_samples <= 1:
      return IsolationTreeNode(size=n_samples)

    feat_idx = np.random.randint(0, n_features)
    feat_min, feat_max = X[:, feat_idx].min(), X[:, feat_idx].max()

    if feat_min == feat_max:
      return IsolationTreeNode(size=n_samples)

    split_val = np.random.uniform(feat_min, feat_max)

    left_mask = X[:, feat_idx] < split_val
    left_child = self._build_tree(X[left_mask], current_depth + 1)
    right_child = self._build_tree(X[~left_mask], current_depth + 1)

    return IsolationTreeNode(
        left=left_child, right=right_child, feature=feat_idx, split=split_val
    )

  def fit(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples = X.shape[0]

    # FIX 1: Store subsample size & compute reference c_factor from training set
    self.subsample_size = min(n_samples, self.max_samples)
    self.max_depth = int(np.ceil(np.log2(max(self.subsample_size, 2))))
    self.c_factor = self._c(self.subsample_size)

    self.trees = []
    for _ in range(self.n_estimators):
      indices = np.random.choice(n_samples, self.subsample_size, replace=False)
      tree = self._build_tree(X[indices], current_depth=0)
      self.trees.append(tree)

    return self

  def _get_path_length(self, x, node, current_depth):
    if node.is_leaf:
      return current_depth + self._c(node.size)

    if x[node.feature] < node.split:
      return self._get_path_length(x, node.left, current_depth + 1)
    else:
      return self._get_path_length(x, node.right, current_depth + 1)

  def compute_anomaly_score(self, X):
    X = np.asarray(X, dtype=np.float64)

    scores = []
    for x in X:
      avg_path = np.mean(
          [self._get_path_length(x, tree, 0) for tree in self.trees]
      )

      # FIX 2: Use training set c_factor instead of calculating c(len(X_test))
      score = 2.0 ** (-avg_path / self.c_factor)
      scores.append(score)

    return np.array(scores)


# TEST SCRIPT

np.random.seed(42)

X_inliers = np.random.normal(loc=2.0, scale=0.5, size=(300, 2))
X_outliers = np.array([[10.0, 10.0], [-5.0, -5.0], [12.0, -2.0]])
X_train = np.vstack([X_inliers, X_outliers])

iforest = IsolationForest(n_estimators=100, max_samples=256)
iforest.fit(X_train)

test_queries = np.array([[2.1, 1.9], [10.0, 10.0], [-5.0, -5.0]])
scores = iforest.compute_anomaly_score(test_queries)

print("=== CORRECTED ISOLATION FOREST ANOMALY SCORES ===")
for i, query in enumerate(test_queries):
  status = "ANOMALY (>0.6)" if scores[i] > 0.6 else "NORMAL (<0.5)"
  print(
      f"Sample {query} -> Anomaly Score: {round(scores[i], 4)} | Status:"
      f" {status}"
  )

=== CORRECTED ISOLATION FOREST ANOMALY SCORES ===
Sample [2.1 1.9] -> Anomaly Score: 0.3604 | Status: NORMAL (<0.5)
Sample [10. 10.] -> Anomaly Score: 0.8728 | Status: ANOMALY (>0.6)
Sample [-5. -5.] -> Anomaly Score: 0.8455 | Status: ANOMALY (>0.6)
